# PA2 Notebook 3: Evaluation

Created from kaggle.com/benchmarks/tasks/new so that `kaggle_benchmarks` is available (no GPU needed,
internet on). Add the output of notebook 2 as input.

The judge (gemini-3.6-flash) compares DPO vs SFT and LoRA vs base using the saved pair files. It sends 5 pairs
per request with a 5 second wait, retries up to 3 times, and saves results after every batch so it can resume.

In [ ]:
# clone the repo (needs internet on)
import os
REPO_URL = "https://github.com/mahamasiff/cs5326-PA2-27100015.git"
REPO_DIR = "/kaggle/working/cs5326-PA2-27100015"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
import subprocess
def sh(cmd):
    """run a shell command, print its output, and stop if it fails"""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode} (see output above): {cmd}")

In [ ]:
import kaggle_benchmarks as kbench
JUDGE = "google/gemini-3.6-flash"  # judge model
print(sorted(kbench.llms))

In [ ]:
# helper to find files from the attached notebook output
import glob
def find_input(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Attach the saved notebook output that contains {pattern}"
    print(pattern, "->", hits[0])
    return hits[0]

In [ ]:
import shutil
os.makedirs("outputs/eval", exist_ok=True)
for name in ("dpo_uf_vs_sft_pairs.jsonl", "dpo_hh_vs_sft_pairs.jsonl", "lora_vs_base_pairs.jsonl"):
    hits = sorted(glob.glob(f"/kaggle/input/**/eval/{name}", recursive=True))
    if hits:
        shutil.copy(hits[0], f"outputs/eval/{name}")
    else:
        # the input wasn't showing up in this notebook, so use the copy of notebook 2's pair files in the repo
        assert os.path.exists(f"outputs/eval/{name}"), f"{name} not found in /kaggle/input or the repo"
        print(f"{name}: using the repository copy of notebook 2's output")
!wc -l outputs/eval/*_pairs.jsonl
# reuse judgments from an earlier run if there are any
for pattern in ("eval/*_judged.jsonl", "eval/*_judged_failures.jsonl"):
    for path in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True):
        shutil.copy(path, "outputs/eval/")
!ls -l outputs/eval

In [ ]:
# test the judge on 2 pairs first
sh("python scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/dpo_uf_vs_sft_pairs.jsonl "
   f"--output_path outputs/eval/smoke_judged.jsonl --judge_model {JUDGE} --limit 2 "
   "--batch_size 5 --sleep_seconds 5 --max_retries 3")
!head -c 2000 outputs/eval/smoke_judged_raw.jsonl
!rm -f outputs/eval/smoke_judged*

## Judging

In [ ]:
for name in ("dpo_uf_vs_sft", "dpo_hh_vs_sft", "lora_vs_base"):
    sh(f"python scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/{name}_pairs.jsonl "
       f"--output_path outputs/eval/{name}_judged.jsonl --judge_model {JUDGE} "
       "--batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments")

In [ ]:
sh("python scripts/collect_results.py --eval_dir outputs/eval --output outputs/eval/results.md")
import shutil
shutil.make_archive("/kaggle/working/evaluation_outputs", "zip", "outputs", "eval")  # zip isn't installed here
!ls -lh /kaggle/working/evaluation_outputs.zip

If some batches fail, run the judging cell again. Pairs that are already judged are skipped.